# Azure OpenAI tool calling for big data

Tool calling lets an OpenAI model request trusted application capabilities while your Spark pipeline remains in control of execution. SynapseML transports tool definitions, structured calls, and tool results; it does not execute arbitrary functions or run an automatic tool loop.

This tutorial shows two complete DataFrame workflows:

- **Chat Completions** with `OpenAIPrompt` for the first turn and `OpenAIChatCompletion` for the assistant/tool-message continuation.
- **Responses API** with `OpenAIPrompt` for the first turn and `OpenAIResponses` for a typed `function_call_output` continuation.


## Prerequisites and notebook import

You need an Azure OpenAI resource, a tool-capable model deployment, and Spark with SynapseML installed. The examples use a `gpt-5.1` deployment; replace it with a compatible deployment available in your resource.

- [Download this notebook](https://github.com/microsoft/SynapseML/blob/master/docs/Explore%20Algorithms/OpenAI/OpenAI_ToolUse.ipynb) by selecting **Raw** and saving the file.
- Import it into [Synapse Analytics](https://learn.microsoft.com/azure/synapse-analytics/spark/apache-spark-development-using-notebooks), [Azure Databricks](https://learn.microsoft.com/azure/databricks/notebooks/notebooks-manage), or [Microsoft Fabric](https://learn.microsoft.com/fabric/data-engineering/how-to-use-notebook).
- Follow the [SynapseML installation guide](https://microsoft.github.io/SynapseML/docs/Get%20Started/Install%20SynapseML/) for your Spark environment.


## Configure Azure OpenAI

Fill in the service and deployment names for your resource. The secret helper is convenient in Microsoft-hosted notebook environments; replace it with your normal secret-management integration when needed.


In [ ]:
from pyspark.sql import functions as F
from synapse.ml.core.platform import find_secret
from synapse.ml.services.openai import (
    OpenAIChatCompletion,
    OpenAIPrompt,
    OpenAIResponses,
)

service_name = "synapseml-openai-3"
deployment_name = "gpt-5.1"
api_version = "2025-04-01-preview"
key = find_secret(
    secret_name="openai-api-key-3", keyvault="mmlspark-build-keys"
)  # Replace this line with your key as a string when appropriate.

system_prompt = "Treat tool results as authoritative and report their exact values."
prompt_template = "What is the weather in {city}? Use the tool."

assert key is not None and service_name is not None and deployment_name is not None

## Tool-calling lifecycle and safety

Each flow follows the same application-controlled lifecycle:

1. Send trusted tool definitions with a batch of prompts.
2. Materialize the paid model turn and read `toolCallsCol`.
3. Treat model-produced arguments as untrusted JSON: parse with a fixed schema, allowlist the function name, and enforce bounds.
4. Resolve the call in ordinary Spark code or an idempotent external service.
5. Return a bounded tool result using the original `call_id`.

Spark plus HTTP is at-least-once. Task retries, executor loss, speculation, or recomputing an unmaterialized lineage can repeat requests and external side effects. Materialize every paid turn, deduplicate by response/call identifiers, and make side effects idempotent by `call_id`.


## Define one trusted function tool and sample data

SynapseML accepts one flat function-tool definition and converts it to the API-specific wire shape. The lookup table below stands in for an application service; no function executes inside the OpenAI transformer.


In [ ]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
        "additionalProperties": False,
    },
    "strict": True,
}

cities = spark.createDataFrame([(1, "Seattle"), (2, "Boston")], ["row_id", "city"])
weather_table = spark.createDataFrame(
    [("Seattle", 20.0, "sunny"), ("Boston", 24.0, "partly cloudy")],
    ["lookup_city", "temp_c", "conditions"],
)

## Chat Completions tool calling

`OpenAIPrompt` remains the easiest first-turn interface for templated DataFrame prompts. Setting `apiType="chat_completions"` sends Chat Completions requests, and `toolCallsCol` projects every returned call into the shared Spark schema.


In [ ]:
chat_turn1 = (
    OpenAIPrompt()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setApiType("chat_completions")
    .setSystemPrompt(system_prompt)
    .setPromptTemplate(prompt_template)
    .setTools([weather_tool])
    .setToolChoice("required")
    .setParallelToolCalls(False)
    .setToolCallsCol("tool_calls")
    .setResponseStructCol("chat_turn1_response")
    .setOutputCol("chat_turn1_text")
    .setErrorCol("chat_turn1_error")
)

chat_asked = chat_turn1.transform(cities).persist()
chat_asked.count()  # Materialize this paid turn before branching.
chat_asked.select(
    "row_id", "city", "chat_turn1_text", "tool_calls", "chat_turn1_error"
).show(truncate=False)

### Validate arguments and resolve the call

The named tool choice and disabled parallel calls make this example produce one call per row. Production pipelines should still handle empty or multiple calls explicitly. Never use `eval`, arbitrary reflection, or model-generated SQL identifiers.


In [ ]:
chat_calls = (
    chat_asked.withColumn("call", F.element_at("tool_calls", 1))
    .withColumn("arguments", F.from_json(F.col("call.arguments"), "city STRING"))
    .where(
        (F.col("call.name") == "get_weather")
        & F.col("arguments.city").isNotNull()
        & (F.length("arguments.city") <= 64)
    )
)

resolved_chat = (
    chat_calls.join(
        weather_table,
        F.col("arguments.city") == F.col("lookup_city"),
        "left",
    )
    .where(F.col("temp_c").isNotNull())
    .select("row_id", "city", "call", "lookup_city", "temp_c", "conditions")
)

### Build Chat assistant and tool messages

A Chat continuation must preserve the original system/user messages, add an assistant message containing `tool_calls`, and add a `role="tool"` message whose `tool_call_id` matches the service call. All message structs in the Spark array use the same schema.


In [ ]:
chat_tool_calls_type = (
    "array<struct<id:string,"
    "function:struct<name:string,arguments:string>,"
    "type:string>>"
)


def chat_message(role, content=None, tool_calls=None, tool_call_id=None):
    content = content if content is not None else F.lit(None).cast("string")
    tool_calls = (
        tool_calls if tool_calls is not None else F.lit(None).cast(chat_tool_calls_type)
    )
    tool_call_id = (
        tool_call_id if tool_call_id is not None else F.lit(None).cast("string")
    )
    return F.struct(
        F.lit(role).alias("role"),
        content.alias("content"),
        F.lit(None).cast("string").alias("name"),
        tool_calls.alias("tool_calls"),
        tool_call_id.alias("tool_call_id"),
    )

In [ ]:
assistant_tool_calls = F.array(
    F.struct(
        F.col("call.call_id").alias("id"),
        F.struct(
            F.col("call.name").alias("name"),
            F.col("call.arguments").alias("arguments"),
        ).alias("function"),
        F.lit("function").alias("type"),
    )
)
chat_tool_result = F.substring(
    F.to_json(
        F.struct(
            F.col("lookup_city").alias("city"),
            F.col("temp_c").alias("temperatureC"),
            F.col("conditions"),
        )
    ),
    1,
    4000,
)

chat_continuations = resolved_chat.select(
    "row_id",
    F.array(
        chat_message("system", F.lit(system_prompt)),
        chat_message(
            "user",
            F.concat(
                F.lit("What is the weather in "),
                F.col("city"),
                F.lit("? Use the tool."),
            ),
        ),
        chat_message("assistant", tool_calls=assistant_tool_calls),
        chat_message(
            "tool",
            content=chat_tool_result,
            tool_call_id=F.col("call.call_id"),
        ),
        chat_message(
            "user",
            F.lit(
                "State the exact Celsius temperature returned by the tool result above."
            ),
        ),
    ).alias("messages"),
)

### Continue with `OpenAIChatCompletion`

Resend the trusted tool definition because tools are request-scoped. `toolChoice="none"` asks the model to produce final text instead of requesting another tool call.


In [ ]:
chat_turn2 = (
    OpenAIChatCompletion()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setMessagesCol("messages")
    .setTools([weather_tool])
    .setToolChoice("none")
    .setMaxCompletionTokens(500)
    .setOutputCol("chat_response")
    .setErrorCol("chat_error")
)

chat_answered = chat_turn2.transform(chat_continuations).persist()
chat_answered.count()
chat_answered.selectExpr(
    "row_id",
    "element_at(chat_response.choices, 1).message.content AS answer",
    "chat_error",
).show(truncate=False)

## Responses API tool calling

The same Spark-facing tool definition also works with `apiType="responses"`. A stored Responses flow returns `response_id`; the continuation sends typed `function_call_output` structs through `functionCallOutputsCol`.


In [ ]:
responses_turn1 = (
    OpenAIPrompt()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setApiType("responses")
    .setSystemPrompt(system_prompt)
    .setPromptTemplate(prompt_template)
    .setTools([weather_tool])
    .setToolChoice("required")
    .setParallelToolCalls(False)
    .setStore(True)
    .setResponseIdCol("response_id")
    .setToolCallsCol("tool_calls")
    .setResponseStructCol("responses_turn1_response")
    .setOutputCol("responses_turn1_text")
    .setErrorCol("responses_turn1_error")
)

responses_asked = responses_turn1.transform(cities).persist()
responses_asked.count()
responses_asked.select(
    "row_id", "city", "response_id", "tool_calls", "responses_turn1_error"
).show(truncate=False)

### Resolve calls and build typed function outputs

`functionCallOutputsCol` expects `ARRAY<STRUCT<call_id, output, status>>`. The output remains an opaque, bounded string. Reuse the same validation and lookup rules rather than trusting model arguments.


In [ ]:
response_calls = (
    responses_asked.select(
        "row_id",
        "response_id",
        F.explode("tool_calls").alias("call"),
    )
    .where(F.col("call.name") == "get_weather")
    .withColumn("arguments", F.from_json(F.col("call.arguments"), "city STRING"))
    .where(F.col("arguments.city").isNotNull() & (F.length("arguments.city") <= 64))
)

resolved_responses = response_calls.join(
    weather_table,
    F.col("arguments.city") == F.col("lookup_city"),
    "left",
).where(F.col("temp_c").isNotNull())

response_outputs = resolved_responses.groupBy("row_id", "response_id").agg(
    F.collect_list(
        F.struct(
            F.col("call.call_id").alias("call_id"),
            F.substring(
                F.to_json(
                    F.struct(
                        F.col("lookup_city").alias("city"),
                        F.col("temp_c").alias("temperatureC"),
                        F.col("conditions"),
                    )
                ),
                1,
                4000,
            ).alias("output"),
            F.lit("completed").alias("status"),
        )
    ).alias("function_outputs")
)

### Continue with `OpenAIResponses`

The stored response identifier reconnects the model's first turn to the tool results. Rows without a response ID or function output are skipped instead of issuing an invalid continuation request.


In [ ]:
responses_turn2 = (
    OpenAIResponses()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setPreviousResponseIdCol("response_id")
    .setFunctionCallOutputsCol("function_outputs")
    .setTools([weather_tool])
    .setToolChoice("none")
    .setOutputCol("responses_response")
    .setErrorCol("responses_error")
)

responses_answered = responses_turn2.transform(response_outputs).persist()
responses_answered.count()
responses_answered.selectExpr(
    "row_id",
    "element_at(filter(responses_response.output, x -> x.type = 'message'), -1).content[0].text AS answer",
    "responses_error",
).show(truncate=False)

## Clean up cached turns


In [ ]:
chat_asked.unpersist()
chat_answered.unpersist()
responses_asked.unpersist()
responses_answered.unpersist()

## Production checklist

- Keep tool definitions in trusted pipeline configuration; do not accept hosted or MCP tool definitions from untrusted rows.
- Parse and validate every model-produced argument with fixed Spark schemas and allowlists.
- Make external effects idempotent by `call_id`, and store enough state to resume after executor or driver failure.
- Materialize every paid turn before branching or displaying it more than once.
- Bound tool-result size and avoid returning secrets, credentials, or unnecessary source data.
- Inspect each transformer's `errorCol` and route failed rows explicitly.

For stateless Responses replay, retry/billing guidance, and more detailed safety notes, see [Quickstart - OpenAI Responses Tool Calling](./Quickstart%20-%20OpenAI%20Responses%20Tool%20Calling.ipynb).
